# 혈액 부족 예측 및 헌혈 운영 최적화 — AI 에이전트

> **UNIST Industrial Operations Management Term Project**  
> Team 2: 노우찬 · 손준영 · 민예지

---

## Phase 0: 데이터 전처리 파이프라인

원시 CSV 7개를 분석 가능한 형태로 정제하고, 에이전트가 사용할 임계값과 피처를 생성합니다.

| 단계 | 파일 | 내용 |
|------|------|------|
| **0-0** | — | 라이브러리 임포트, 경로 설정 |
| **0-1** | 파일 5 | 일별 혈액 보유량 → 날짜 인덱스 시계열 |
| **0-2** | 파일 6 | 월별 혈액 보유량(제제별) → Long format |
| **0-3** | 파일 2 | 월별 헌혈통계 → 전국 합계 시계열 |
| **0-4** | 파일 3 | 연도별 헌혈자 수 → 깔끔한 DataFrame |
| **0-5** | — | 피처 엔지니어링 (계절·위험시즌 변수) |
| **0-6** | — | 안전재고 임계값 설정 및 위험 등급 함수 |
| **0-7** | — | 처리된 파일 저장 및 최종 요약 |

---
### Phase 0-0: 라이브러리 임포트 및 경로 설정

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

BASE      = r'C:\Users\shdnc\OneDrive\Desktop\UNIST\2026 생운관\File_maker_Project'
PROCESSED = os.path.join(BASE, 'processed')
os.makedirs(PROCESSED, exist_ok=True)

print('✅ 라이브러리 로드 완료')
print(f'📁 원본 데이터: {BASE}')
print(f'📁 저장 경로:   {PROCESSED}')

---
### Phase 0-1: 일별 혈액 보유량 (파일 5)

- **원본**: `5.Blood_retention_trends_for_blood_transfusion(Day).csv`
- **형태**: Wide format — 날짜(366행) × 연도(2015–2025)
- **목적**: 에이전트 예측 모듈의 핵심 시계열 → Long format으로 변환

In [ ]:
df5_raw = pd.read_csv(
    f'{BASE}/5.Blood_retention_trends_for_blood_transfusion(Day).csv',
    encoding='cp949', header=None
)

# 헤더 행에서 연도 목록 추출
years     = df5_raw.iloc[0, 1:].astype(str).str.strip().tolist()   # ['2015', ..., '2025']
date_strs = df5_raw.iloc[1:, 0].tolist()                           # ['01월 01일', ...]

# Wide → Long 변환
records = []
for col_offset, yr in enumerate(years):
    col_idx = col_offset + 1
    for row_offset, date_str in enumerate(date_strs):
        val_raw = df5_raw.iloc[row_offset + 1, col_idx]
        val = pd.to_numeric(str(val_raw).replace(',', ''), errors='coerce')

        # '01월 01일' → '2015-01-01'
        md = str(date_str).strip().replace('월 ', '-').replace('일', '').strip()
        try:
            date = pd.to_datetime(f'{yr}-{md}', format='%Y-%m-%d')
            if pd.notna(val):
                records.append({'date': date, 'year': int(yr), 'inventory': val})
        except Exception:
            pass  # 윤년 아닌 해의 2월 29일 등 무시

daily_inventory = (
    pd.DataFrame(records)
    .dropna(subset=['inventory'])
    .sort_values('date')
    .reset_index(drop=True)
)
daily_inventory['inventory'] = daily_inventory['inventory'].astype(int)

print(f'✅ 일별 보유량: {len(daily_inventory):,}행')
print(f'   기간: {daily_inventory["date"].min().date()} ~ {daily_inventory["date"].max().date()}')
print(f'   범위: {daily_inventory["inventory"].min():,} ~ {daily_inventory["inventory"].max():,} unit')
print()
print('연도별 평균 보유량 (unit):')
print(daily_inventory.groupby('year')['inventory'].mean().round(0).astype(int).to_string())

In [ ]:
# 전체 일별 시계열 시각화
ts = daily_inventory.set_index('date')['inventory']

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.index, ts.values, color='steelblue', linewidth=0.7, alpha=0.8)
ax.axhline(y=20000, color='orange', linestyle='--', linewidth=1.2, label='주의선 (20,000)')
ax.axhline(y=15000, color='red',    linestyle='--', linewidth=1.2, label='경고선 (15,000)')
ax.axhline(y=10000, color='darkred',linestyle=':',  linewidth=1.2, label='위기선 (10,000)')
ax.set_title('일별 혈액 보유량 (2015–2025)', fontsize=13, fontweight='bold')
ax.set_ylabel('보유량 (unit)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
### Phase 0-2: 월별 혈액 보유량 — 제제별 (파일 6)

- **원본**: `6.Blood_retention_trends_for_blood_transfusion(Month).csv`
- **형태**: 월(12행) × [연도 × 제제](44열)
- **제제 4종**: 농축적혈구(RBC) · 농축혈소판(PLT) · 신선동결혈장(FFP) · 백혈구여과제거성분채혈혈소판(SDP)
- **목적**: 제제별 부족 위험도 개별 평가 (혈소판 유효기간 5일 → 별도 모니터링)

In [ ]:
df6_raw = pd.read_csv(
    f'{BASE}/6.Blood_retention_trends_for_blood_transfusion(Month).csv',
    encoding='cp949', header=None
)

# 헤더 추출
years_row = df6_raw.iloc[0, 1:].astype(str).str.strip().tolist()
types_row = df6_raw.iloc[1, 1:].astype(str).str.strip().tolist()
months    = df6_raw.iloc[2:, 0].tolist()       # ['1월', ..., '12월']
data_mat  = df6_raw.iloc[2:, 1:].reset_index(drop=True)

# 영문 코드 매핑
COMP_CODE = {
    '농축적혈구':              'RBC',
    '농축혈소판':              'PLT',
    '신선동결혈장':             'FFP',
    '백혈구여과제거성분채혈혈소판': 'SDP',
}

records = []
for col_offset, (yr, comp) in enumerate(zip(years_row, types_row)):
    for row_offset, mo_str in enumerate(months):
        val_raw  = data_mat.iloc[row_offset, col_offset]
        val      = pd.to_numeric(str(val_raw).replace(',', ''), errors='coerce')
        month_num = int(str(mo_str).replace('월', '').strip())
        records.append({
            'year':           int(str(yr).strip()),
            'month':          month_num,
            'component':      comp,
            'component_code': COMP_CODE.get(comp, comp),
            'inventory':      val,
        })

monthly_by_type = (
    pd.DataFrame(records)
    .dropna(subset=['inventory'])
    .reset_index(drop=True)
)
monthly_by_type['date'] = pd.to_datetime(
    monthly_by_type.apply(
        lambda r: f"{int(r['year'])}-{int(r['month']):02d}-01", axis=1
    )
)

print(f'✅ 제제별 월별 보유량: {len(monthly_by_type):,}행')
print(f'   제제 종류: {monthly_by_type["component_code"].unique().tolist()}')
print(f'   기간: {monthly_by_type["year"].min()} ~ {monthly_by_type["year"].max()}')
print()

# 제제별 연평균 보유량
pivot = monthly_by_type.groupby(['year', 'component_code'])['inventory'].mean().round(0).unstack()
print('제제별 연평균 보유량:')
print(pivot.astype(int).to_string())

In [ ]:
# 제제별 월별 보유량 트렌드 (2020–2025)
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=False)
codes  = ['RBC', 'PLT', 'FFP', 'SDP']
titles = ['농축적혈구 (RBC)', '농축혈소판 (PLT)', '신선동결혈장 (FFP)', '성분채혈혈소판 (SDP)']
colors = ['steelblue', 'tomato', 'seagreen', 'mediumpurple']

for ax, code, title, color in zip(axes.flat, codes, titles, colors):
    df_sub = monthly_by_type[
        (monthly_by_type['component_code'] == code) &
        (monthly_by_type['year'] >= 2020)
    ].sort_values('date')
    ax.plot(df_sub['date'], df_sub['inventory'], color=color, linewidth=1.2)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('제제별 월별 보유량 추이 (2020–2025)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
### Phase 0-3: 월별 헌혈통계 — 전국 합계 (파일 2)

- **원본**: `2.Month_blood.csv`
- **형태**: 3행 복합 헤더 + 지역·성별별 데이터
- **추출**: 전국 합계(계) 행 — 월별 헌혈 건수 시계열
- **목적**: 헌혈량과 보유량의 상관 분석, 계절 헌혈 패턴 파악

In [ ]:
df2_raw = pd.read_csv(
    f'{BASE}/2.Month_blood.csv',
    encoding='cp949', header=None
)

# 복합 헤더 파싱 (행 0=연도, 행 1=월, 행 2=지표)
years_row   = df2_raw.iloc[0, 3:].tolist()
months_row  = df2_raw.iloc[1, 3:].tolist()
metrics_row = df2_raw.iloc[2, 3:].tolist()

# 전국 합계 행 (행 3: 혈액원=합계, 소계, 성별=계)
national_vals = df2_raw.iloc[3, 3:].tolist()

records = []
for yr, mo, metric, val in zip(years_row, months_row, metrics_row, national_vals):
    yr_str     = str(yr).strip()
    mo_str     = str(mo).strip()
    metric_str = str(metric).strip()

    # 월별 실적(건)만 추출 — 연간합계 열과 구성비 열 제외
    if metric_str == '실적 (건)' and mo_str not in ('합계', 'nan', ''):
        try:
            month_num = int(mo_str.replace('월', '').strip())
            val_clean = int(pd.to_numeric(str(val).replace(',', ''), errors='coerce'))
            records.append({'year': int(yr_str), 'month': month_num, 'donation': val_clean})
        except Exception:
            pass

monthly_donation = pd.DataFrame(records).dropna().reset_index(drop=True)
monthly_donation['date'] = pd.to_datetime(
    monthly_donation.apply(
        lambda r: f"{int(r['year'])}-{int(r['month']):02d}-01", axis=1
    )
)

print(f'✅ 월별 헌혈통계(전국): {len(monthly_donation):,}행')
print(f'   기간: {monthly_donation["year"].min()}년 ~ {monthly_donation["year"].max()}년')
print(f'   월 평균 헌혈 건수: {monthly_donation["donation"].mean():,.0f}건')
print()

# 월별 평균 헌혈량 (계절 패턴 확인)
monthly_avg = monthly_donation.groupby('month')['donation'].mean().round(0).astype(int)
print('월별 평균 헌혈량 (2015–2025 평균):')
for m, v in monthly_avg.items():
    bar = '█' * (v // 10000)
    print(f'  {m:2d}월: {v:,}건  {bar}')

In [ ]:
# 월별 헌혈량 시계열
md_ts = monthly_donation.set_index('date')['donation'].sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(md_ts.index, md_ts.values, color='darkorange', linewidth=0.9)
ax.fill_between(md_ts.index, md_ts.values, alpha=0.15, color='darkorange')
ax.set_title('월별 전국 헌혈 건수 (2015–2025)', fontsize=13, fontweight='bold')
ax.set_ylabel('헌혈 건수')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
### Phase 0-4: 연도별 헌혈자 수 (파일 3)

- **원본**: `3.Number_of_blood_donors.csv`
- **형태**: 깔끔한 단일 헤더, 2015–2025 연간 요약
- **목적**: 헌혈률 장기 추세 파악 (캠페인 효과 기준선)

In [ ]:
df3_raw = pd.read_csv(
    f'{BASE}/3.Number_of_blood_donors.csv',
    encoding='cp949', header=None
)

# 행 0이 헤더
col_names = df3_raw.iloc[0].tolist()
yearly_donors = df3_raw.iloc[1:].copy()
yearly_donors.columns = col_names
yearly_donors = yearly_donors.rename(columns={col_names[0]: 'year'})
yearly_donors['year'] = yearly_donors['year'].astype(int)

# 숫자 컬럼 변환
for col in col_names[1:]:
    yearly_donors[col] = pd.to_numeric(
        yearly_donors[col].astype(str).str.replace(',', ''),
        errors='coerce'
    )
yearly_donors = yearly_donors.reset_index(drop=True)

print('✅ 연도별 헌혈자 통계:')
print(yearly_donors[['year', '총 헌혈실적 (건)', '헌혈률 (%)', '헌혈자 실인원 (명)', '실제 국민 헌혈률 (%)']].to_string(index=False))

In [ ]:
# 헌혈률 장기 추세
fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.bar(yearly_donors['year'], yearly_donors['총 헌혈실적 (건)'],
        color='steelblue', alpha=0.6, label='총 헌혈실적 (건)')
ax1.set_ylabel('총 헌혈실적 (건)', color='steelblue')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2 = ax1.twinx()
ax2.plot(yearly_donors['year'], yearly_donors['실제 국민 헌혈률 (%)'],
         color='tomato', marker='o', linewidth=2, label='실제 국민 헌혈률 (%)')
ax2.set_ylabel('실제 국민 헌혈률 (%)', color='tomato')

ax1.set_title('연도별 총 헌혈실적 및 헌혈률 (2015–2025)', fontsize=12, fontweight='bold')
ax1.set_xticks(yearly_donors['year'])
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

---
### Phase 0-5: 피처 엔지니어링

`daily_inventory`에 에이전트가 사용할 파생 변수를 추가합니다.

| 변수 | 설명 |
|------|------|
| `month`, `day_of_year` | 날짜 기반 변수 |
| `season` | 봄·여름·가을·겨울 |
| `is_risk_season` | 3–4월(봄 위기), 10–12월(가을·겨울 위기) = 1 |
| `ma7` | 7일 이동평균 (단기 추세) |
| `ma30` | 30일 이동평균 (중기 추세) |
| `daily_change` | 전일 대비 변화량 |
| `yoy_inventory` | 전년 동일 날짜 보유량 (Year-over-Year) |
| `yoy_diff` | 전년 대비 차이 |

In [ ]:
di = daily_inventory.copy()
di = di.set_index('date').sort_index()

# 날짜 파생 변수
di['month']       = di.index.month
di['day_of_year'] = di.index.dayofyear
di['weekday']     = di.index.weekday   # 0=월, 6=일

# 계절
season_map = {12: '겨울', 1: '겨울', 2: '겨울',
               3: '봄',   4: '봄',   5: '봄',
               6: '여름', 7: '여름', 8: '여름',
               9: '가을', 10: '가을', 11: '가을'}
di['season'] = di['month'].map(season_map)

# 위험 시즌 플래그 (역사적 위기 집중 구간)
di['is_risk_season'] = di['month'].isin([3, 4, 10, 11, 12]).astype(int)

# 이동평균
di['ma7']  = di['inventory'].rolling(window=7,  min_periods=1).mean().round(0)
di['ma30'] = di['inventory'].rolling(window=30, min_periods=1).mean().round(0)

# 일별 변화량
di['daily_change'] = di['inventory'].diff()

# 전년 동일 날짜 보유량 (YoY 비교)
# 2016-02-29처럼 윤년 날짜를 비윤년 전년도로 치환할 수 없는 경우 NaN 처리
di_ts = di['inventory'].copy()
yoy_vals = []
for idx in di.index:
    try:
        prev_year_date = idx.replace(year=idx.year - 1)
    except ValueError:
        prev_year_date = pd.NaT
    if pd.notna(prev_year_date) and prev_year_date in di_ts.index:
        yoy_vals.append(di_ts[prev_year_date])
    else:
        yoy_vals.append(np.nan)

di['yoy_inventory'] = yoy_vals
di['yoy_diff']      = di['inventory'] - di['yoy_inventory']

daily_inventory_feat = di.reset_index()

print(f'✅ 피처 엔지니어링 완료: {daily_inventory_feat.shape}')
print(f'   추가된 컬럼: {[c for c in daily_inventory_feat.columns if c not in ["date","year","inventory"]]}')
print()

# 위험 시즌 vs 정상 시즌 비교
risk_avg   = di[di['is_risk_season'] == 1]['inventory'].mean()
normal_avg = di[di['is_risk_season'] == 0]['inventory'].mean()
print(f'위험 시즌 평균 보유량: {risk_avg:,.0f} unit  (3·4·10·11·12월)')
print(f'일반 시즌 평균 보유량: {normal_avg:,.0f} unit  (5·6·7·8·9월)')
print(f'차이: {risk_avg - normal_avg:,.0f} unit ({(risk_avg/normal_avg - 1)*100:.1f}%)')

In [ ]:
# 계절별 보유량 박스플롯 (계절 패턴 시각화)
season_order = ['봄', '여름', '가을', '겨울']
season_data  = [di[di['season'] == s]['inventory'].values for s in season_order]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 왼쪽: 계절별 박스플롯
bp = axes[0].boxplot(season_data, labels=season_order, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
colors_box = ['#ff9999', '#99ccff', '#ffcc99', '#cc99ff']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].axhline(y=20000, color='orange', linestyle='--', linewidth=1, label='주의선')
axes[0].axhline(y=15000, color='red',    linestyle='--', linewidth=1, label='경고선')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].set_title('계절별 혈액 보유량 분포', fontsize=11, fontweight='bold')
axes[0].set_ylabel('보유량 (unit)')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# 오른쪽: 월별 평균 보유량 (위험구간 강조)
monthly_mean = di.groupby('month')['inventory'].mean()
bar_colors = ['#ff7777' if m in [3, 4, 10, 11, 12] else '#6699cc' for m in range(1, 13)]
axes[1].bar(range(1, 13), monthly_mean.values, color=bar_colors, alpha=0.8)
axes[1].axhline(y=20000, color='orange', linestyle='--', linewidth=1.2, label='주의선 (20,000)')
axes[1].axhline(y=15000, color='red',    linestyle='--', linewidth=1.2, label='경고선 (15,000)')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels([f'{m}월' for m in range(1, 13)])
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].set_title('월별 평균 보유량 (빨강=위험 시즌)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('평균 보유량 (unit)')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
### Phase 0-6: 안전재고 임계값 설정

역사적 데이터 분석을 근거로 4단계 위험 등급 임계값을 설정합니다.

| 등급 | 임계값 | 근거 |
|------|--------|------|
| 🟢 NORMAL  | ≥ 20,000 unit | 연평균 최솟값 이상 (안전구간) |
| 🟡 CAUTION | 15,000–19,999 | 위기 전 완충구간 |
| 🟠 WARNING | 10,000–14,999 | 대부분 위기연도 최솟값 범위 |
| 🔴 CRITICAL | < 10,000 | 2016년 9,640 unit 역대 최저치 근방 |

In [ ]:
THRESHOLDS = {
    'CRITICAL': 10000,
    'WARNING':  15000,
    'CAUTION':  20000,
    'levels': [
        {'level': 'CRITICAL', 'label': '🔴 위기',  'below': 10000,
         'action': '긴급 전 채널 캠페인 (+15% 목표)', 'color': '#cc0000'},
        {'level': 'WARNING',  'label': '🟠 경고',  'below': 15000,
         'action': 'SNS 캠페인 + 단체헌혈 요청 (+10% 목표)', 'color': '#ff6600'},
        {'level': 'CAUTION',  'label': '🟡 주의',  'below': 20000,
         'action': '정기헌혈자 SMS 발송 (+5% 목표)', 'color': '#ffaa00'},
        {'level': 'NORMAL',   'label': '🟢 정상',  'below': None,
         'action': '모니터링 유지', 'color': '#009900'},
    ],
    'historical_min': {
        '2015': {'value': 12425, 'date': '10월 14일'},
        '2016': {'value': 9640,  'date': '01월 06일'},
        '2017': {'value': 14580, 'date': '04월 20일'},
        '2018': {'value': 11071, 'date': '10월 11일'},
        '2019': {'value': 11029, 'date': '04월 24일'},
        '2020': {'value': 10110, 'date': '12월 17일'},
        '2021': {'value': 10555, 'date': '04월 22일'},
        '2022': {'value': 12405, 'date': '04월 07일'},
        '2023': {'value': 12785, 'date': '03월 09일'},
        '2024': {'value': 15062, 'date': '02월 16일'},
        '2025': {'value': 12286, 'date': '04월 18일'},
    }
}

# 위험 등급 분류 함수 (에이전트가 직접 호출)
def classify_risk(inventory: int) -> dict:
    """보유량을 받아 위험 등급·레이블·권고 액션을 반환합니다."""
    if inventory < THRESHOLDS['CRITICAL']:
        level = 'CRITICAL'
    elif inventory < THRESHOLDS['WARNING']:
        level = 'WARNING'
    elif inventory < THRESHOLDS['CAUTION']:
        level = 'CAUTION'
    else:
        level = 'NORMAL'

    info = next(l for l in THRESHOLDS['levels'] if l['level'] == level)
    return {
        'level':     level,
        'label':     info['label'],
        'action':    info['action'],
        'inventory': inventory,
    }

# 저장
threshold_path = os.path.join(PROCESSED, 'thresholds.json')
with open(threshold_path, 'w', encoding='utf-8') as f:
    json.dump(THRESHOLDS, f, ensure_ascii=False, indent=2)

print('✅ 임계값 설정 및 저장 완료')
print()
print('위험 등급 분류 함수 테스트:')
for test_val in [25000, 18000, 12000, 8000]:
    r = classify_risk(test_val)
    print(f'  {test_val:,} unit → {r["label"]}  ({r["action"]})')

In [ ]:
# 역대 연도별 최저 보유량 vs 임계값 시각화
hist_min = THRESHOLDS['historical_min']
yrs  = list(hist_min.keys())
vals = [hist_min[y]['value'] for y in yrs]
bar_colors = [
    '#cc0000' if v < 10000 else
    '#ff6600' if v < 15000 else
    '#ffaa00' if v < 20000 else
    '#009900'
    for v in vals
]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(yrs, vals, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.axhline(y=20000, color='#ffaa00', linestyle='--', linewidth=1.5, label='주의선 (20,000)')
ax.axhline(y=15000, color='#ff6600', linestyle='--', linewidth=1.5, label='경고선 (15,000)')
ax.axhline(y=10000, color='#cc0000', linestyle=':',  linewidth=1.5, label='위기선 (10,000)')

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
            f'{val:,}', ha='center', va='bottom', fontsize=8.5)

ax.set_title('연도별 혈액 보유량 최저치 vs 위험 임계값', fontsize=12, fontweight='bold')
ax.set_ylabel('최저 보유량 (unit)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(vals) * 1.15)
plt.tight_layout()
plt.show()

---
### Phase 0-7: 처리된 파일 저장 및 최종 요약

In [ ]:
# 저장
save_map = {
    'daily_inventory.csv':           daily_inventory_feat,
    'monthly_inventory_by_type.csv': monthly_by_type,
    'monthly_donation.csv':          monthly_donation,
    'yearly_donors.csv':             yearly_donors,
}

for fname, df in save_map.items():
    path = os.path.join(PROCESSED, fname)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'💾 저장: {fname}  ({df.shape[0]:,}행 × {df.shape[1]}열)')

print(f'💾 저장: thresholds.json')
print()
print('=' * 55)
print('  Phase 0 완료 — 데이터 전처리 파이프라인 요약')
print('=' * 55)

summary = [
    ('daily_inventory',           len(daily_inventory_feat),  '일별 보유량 + 피처 (2015–2025)'),
    ('monthly_inventory_by_type', len(monthly_by_type),       '제제별 월별 보유량 (RBC/PLT/FFP/SDP)'),
    ('monthly_donation',          len(monthly_donation),      '전국 월별 헌혈 건수'),
    ('yearly_donors',             len(yearly_donors),         '연도별 헌혈자·헌혈률'),
    ('thresholds.json',           4,                          '위험 등급: NORMAL/CAUTION/WARNING/CRITICAL'),
]

print(f'  {"데이터셋":<32} {"행 수":>6}  설명')
print('-' * 55)
for name, rows, desc in summary:
    print(f'  {name:<32} {rows:>6}  {desc}')

print()
print('► 다음 단계: Phase 1 — Forecasting Module (SMA · Holt-Winters)')

---

## 참고: KOSIS API (실시간 데이터 수집)

CSV가 아닌 KOSIS Open API로 최신 데이터를 직접 받아오는 코드입니다.  
Phase 1 이후 에이전트의 Sensing 단계에서 사용합니다.

In [ ]:
import requests
import pandas as pd

# KOSIS에서 생성된 URL 그대로 사용
url = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

params = {
    "method":       "getList",
    "apiKey":       "MDRiYjNmYWZlOTk1MDk0MGEzOTRkMmY2MTY1MmU3ZDc=",
    "itmId":        "T001 T002 ",
    "objL1":        "ALL",
    "objL2":        "ALL",
    "objL3":        "ALL",
    "objL4":        "",
    "objL5":        "",
    "objL6":        "",
    "objL7":        "",
    "objL8":        "",
    "format":       "json",
    "jsonVD":       "Y",
    "prdSe":        "M",
    "newEstPrdCnt": "10",
    "prdInterval":  "1",
    "orgId":        "445",
    "tblId":        "DT_445001_002"
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data)
print(df.head())
print(df.columns.tolist())

In [ ]:
import requests
import pandas as pd

API_KEY = "MDRiYjNmYWZlOTk1MDk0MGEzOTRkMmY2MTY1MmU3ZDc="
url = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

params = {
    "method":       "getList",
    "apiKey":       API_KEY,
    "itmId":        "T001+T002+",
    "objL1":        "ALL",
    "objL2":        "ALL",
    "objL3":        "ALL",
    "objL4":        "", "objL5": "", "objL6": "", "objL7": "", "objL8": "",
    "format":       "json",
    "jsonVD":       "Y",
    "prdSe":        "M",
    "newEstPrdCnt": "120",
    "prdInterval":  "1",
    "orgId":        "445",
    "tblId":        "DT_445001_002"
}

response = requests.get(url, params=params)
df = pd.DataFrame(response.json())

df_clean = df[['PRD_DE', 'C1_NM', 'C2_NM', 'C3_NM', 'ITM_NM', 'DT', 'UNIT_NM']].copy()
df_clean['DT'] = pd.to_numeric(df_clean['DT'], errors='coerce')

print("샘플 데이터:")
print(df_clean.head(20))
print("\n월 범위:", sorted(df_clean['PRD_DE'].unique())[:10], "...")